In [ ]:
from pyspark.sql import SparkSession
import os as os

OIDC_TOKEN = os.environ["OIDC_TOKEN"]
spark = (SparkSession.builder.appName("test").remote("sc://spark-connector-0-driver-svc:15002/;x-afon-token=" + OIDC_TOKEN)
# AWS 
.config("spark.sql.catalog.test_aws.scope", "openid")
.config("spark.sql.catalog.test_aws.header.X-Iceberg-Access-Delegation", "vended-credentials")
.config("spark.sql.catalog.test_aws", "org.apache.iceberg.spark.SparkCatalog")
.config("spark.sql.catalog.test_aws.type", "rest")
.config("spark.sql.catalog.test_aws.uri", "https://emryspolaris.click/api/catalog")
.config("spark.sql.catalog.test_aws.cache-enabled", "false")
.config("spark.sql.catalog.test_aws.warehouse", "aws")
.config("spark.sql.catalog.test_aws.token", OIDC_TOKEN)

).getOrCreate()

print("Finished initializing spark session")


This needs the smoke test that writes the properties to a test table to run:

In [ ]:
#
# AWS
#
df_read = spark.read.table("test_aws.admin.deployments")
df_read.show()

In [ ]:
from trino.auth import JWTAuthentication
from trino.dbapi import connect

conn = connect(
    host="trino",
    port=8443,
    catalog="aws",
    schema="admin",
    http_scheme="https",
    auth=JWTAuthentication(OIDC_TOKEN),
    extra_credential=[("token", OIDC_TOKEN)],
    verify="/etc/trino-tls/ca.crt",
    # verify=False,  # Uncomment if using self-signed internal TLS certificates
)

cursor = conn.cursor()
cursor.execute("SELECT current_catalog")
print(cursor.fetchone()) # this should work

cursor.execute("SELECT * FROM aws.admin.deployments LIMIT 5")
print(cursor.fetchall())

We should be allowed to run Python UDFs but not to use them to contact the connector

In [ ]:
from pyspark.sql.functions import udf

host = "spark-connector-0-driver-svc"

@udf("string")
def probe(_):
    import urllib.request
    try:
        with urllib.request.urlopen(f"http://{host}:4040/api/v1/applications", timeout=5) as r:
            return f"{r.status} {r.read(300)!r}"
    except Exception as e:
        return repr(e)

spark.range(1).select(probe("id")).show(truncate=False)